In [ ]:
!pip -q install rdkit

In [ ]:

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

from sklearn.model_selection import StratifiedKFold, train_test_split, learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc

SEED = 42
N_SPLITS = 5

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)


In [ ]:


df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.dropna(subset=["smiles", "HIV_active"]).reset_index(drop=True)

print("Shape:", df.shape)
print(df["HIV_active"].value_counts())
df.head()


In [ ]:


def smiles_to_morgan(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((nBits,), dtype=np.float32)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    arr = np.zeros((nBits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

X = np.array([smiles_to_morgan(s) for s in df["smiles"]], dtype=np.float32)
y = df["HIV_active"].astype(int).values

print("X shape:", X.shape, "| y shape:", y.shape)


In [ ]:

def evaluate_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def run_fold(X_train_full, y_train_full, X_test, y_test, fold_seed):
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.20,
        random_state=fold_seed,
        stratify=y_train_full
    )

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc = scaler.transform(X_val)
    X_test_sc = scaler.transform(X_test)

    model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=fold_seed)
    model.fit(X_train_sc, y_train)

    val_prob = model.predict_proba(X_val_sc)[:, 1]
    fpr_val, tpr_val, thresholds = roc_curve(y_val, val_prob)
    youden = tpr_val - fpr_val
    best_idx = np.argmax(youden)
    best_threshold = thresholds[best_idx]

    test_prob = model.predict_proba(X_test_sc)[:, 1]
    test_pred = (test_prob >= best_threshold).astype(int)
    metrics = evaluate_metrics(y_test, test_pred, test_prob)
    metrics["threshold"] = float(best_threshold)

    fpr_test, tpr_test, _ = roc_curve(y_test, test_prob)
    metrics["roc_fpr"] = fpr_test
    metrics["roc_tpr"] = tpr_test
    return metrics


In [ ]:


skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results = []
roc_data = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    print(f"==================== FOLD {fold} ====================")
    set_seed(SEED + fold)

    X_train_full, X_test = X[train_idx], X[test_idx]
    y_train_full, y_test = y[train_idx], y[test_idx]

    metrics = run_fold(X_train_full, y_train_full, X_test, y_test, fold_seed=SEED + fold)

    roc_data.append((metrics["roc_fpr"], metrics["roc_tpr"]))
    metrics_row = {k: v for k, v in metrics.items() if k not in ["roc_fpr", "roc_tpr"]}
    metrics_row["fold"] = fold
    fold_results.append(metrics_row)

    printable = {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics_row.items()}
    print(printable)

results_df = pd.DataFrame(fold_results)
results_df = results_df[["fold", "accuracy", "precision", "recall", "f1", "roc_auc", "threshold"]]
results_df


In [ ]:

results_df.to_csv("Mol2VecSVM_fold_results_with_validation.csv", index=False)

summary_rows = []
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("Mol2VecSVM_summary_results_with_validation.csv", index=False)

print("Fold bazlı sonuçlar:")
display(results_df)

print("\nÖzet sonuçlar:")
display(summary_df)


In [ ]:


mean_fpr = np.linspace(0, 1, 200)
interp_tprs = []
fold_aucs = []

for fpr_i, tpr_i in roc_data:
    interp_tpr = np.interp(mean_fpr, fpr_i, tpr_i)
    interp_tpr[0] = 0.0
    interp_tprs.append(interp_tpr)
    fold_aucs.append(auc(fpr_i, tpr_i))

mean_tpr = np.mean(interp_tprs, axis=0)
std_tpr = np.std(interp_tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = np.mean(fold_aucs)
std_auc = np.std(fold_aucs, ddof=1)

plt.figure(figsize=(7, 6))
for i, (fpr_i, tpr_i) in enumerate(roc_data, start=1):
    plt.plot(fpr_i, tpr_i, alpha=0.25, label=f"Fold {i}")

plt.plot(mean_fpr, mean_tpr, linewidth=2, label=f"Mean ROC (AUC = {mean_auc:.3f} ± {std_auc:.3f})")
plt.fill_between(mean_fpr, np.maximum(mean_tpr - std_tpr, 0), np.minimum(mean_tpr + std_tpr, 1), alpha=0.2, label="±1 std")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Mol2Vec/Fingerprint + SVM Mean ROC Curve (5-Fold)")
plt.legend(loc="lower right", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Mol2VecSVM_mean_ROC_curve.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale", probability=False, random_state=SEED))
])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


In [ ]:


train_sizes, train_scores, valid_scores = learning_curve(
    estimator=pipeline,
    X=X,
    y=y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    shuffle=True,
    random_state=SEED
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1, ddof=1)
valid_mean = valid_scores.mean(axis=1)
valid_std = valid_scores.std(axis=1, ddof=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, marker="o", label="Train ROC-AUC")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
plt.plot(train_sizes, valid_mean, marker="s", label="Validation ROC-AUC")
plt.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.2)
plt.xlabel("Training Set Size")
plt.ylabel("ROC-AUC")
plt.title("Learning Curve - Mol2Vec/Fingerprint + SVM")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Mol2VecSVM_learning_curve.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:


param_range = np.logspace(-2, 2, 7)

train_scores_v, valid_scores_v = validation_curve(
    estimator=pipeline,
    X=X,
    y=y,
    param_name="svc__C",
    param_range=param_range,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

train_mean_v = train_scores_v.mean(axis=1)
train_std_v = train_scores_v.std(axis=1, ddof=1)
valid_mean_v = valid_scores_v.mean(axis=1)
valid_std_v = valid_scores_v.std(axis=1, ddof=1)

val_curve_df = pd.DataFrame({
    "C": param_range,
    "Train_Mean_ROC_AUC": train_mean_v,
    "Train_Std": train_std_v,
    "Validation_Mean_ROC_AUC": valid_mean_v,
    "Validation_Std": valid_std_v
})
val_curve_df.to_csv("Mol2VecSVM_validation_curve_C.csv", index=False)
display(val_curve_df)

plt.figure(figsize=(8, 5))
plt.semilogx(param_range, train_mean_v, marker="o", label="Train ROC-AUC")
plt.fill_between(param_range, train_mean_v - train_std_v, train_mean_v + train_std_v, alpha=0.2)
plt.semilogx(param_range, valid_mean_v, marker="s", label="Validation ROC-AUC")
plt.fill_between(param_range, valid_mean_v - valid_std_v, valid_mean_v + valid_std_v, alpha=0.2)

best_idx = np.argmax(valid_mean_v)
best_c = param_range[best_idx]
best_val_auc = valid_mean_v[best_idx]
plt.axvline(best_c, linestyle="--", color="gray", alpha=0.7, label=f"Best C = {best_c:.3g}")

plt.xlabel("C Parameter (log scale)")
plt.ylabel("ROC-AUC")
plt.title("Validation Curve - Mol2Vec/Fingerprint + SVM")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("Mol2VecSVM_validation_curve_C.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Best C according to validation mean ROC-AUC: {best_c:.4g}")
print(f"Best validation ROC-AUC: {best_val_auc:.4f}")


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os, shutil

save_dir = "/content/drive/MyDrive/HIV_Project"
os.makedirs(save_dir, exist_ok=True)

files_to_save = [
    "Mol2VecSVM_fold_results_with_validation.csv",
    "Mol2VecSVM_summary_results_with_validation.csv",
    "Mol2VecSVM_mean_ROC_curve.png",
    "Mol2VecSVM_learning_curve.png",
    "Mol2VecSVM_validation_curve_C.png",
    "Mol2VecSVM_validation_curve_C.csv"
]

for fname in files_to_save:
    if os.path.exists(fname):
        dst = os.path.join(save_dir, fname)
        shutil.copy(fname, dst)
        print(f"Saved: {dst}")
    else:
        print(f"File not found: {fname}")
